# Starting a job — does chat time move out of work hours? (week 4, distributions)

Builds the Analysis 11 chart from `analysis-log.md`, for the week-4 distributions
assignment (`week4-distribution-requirements.md`).

**Question (Stage 1):** does starting a job push a person's weekday chatting from
work hours to the evening — or do people just keep chatting during work, as
Analysis 8 hinted?

**Primary statistic, fixed before looking:** per active weekday, the share of that
day's messages sent 09:00–17:00, averaged per period; *(starter's after − before)
− (control's after − before)*. Prediction: negative.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wa_analyzer.data import load_own_chat

own = load_own_chat()
# same fix as 06/07: `timestamp` is stored as strings; the `+00:00` is only a label --
# these are the raw export's local clock times (checked against `_chat.txt`).
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)

weekdays = own[own["timestamp"].dt.weekday < 5].copy()  # a job runs Mon-Fri
weekdays["date"] = weekdays["timestamp"].dt.normalize()

BUCKETS = ["night\n00–06", "morning\n06–09", "work hours\n09–17", "evening\n17–24"]
WORK = BUCKETS[2]
weekdays["bucket"] = pd.cut(
    weekdays["timestamp"].dt.hour, [0, 6, 9, 17, 24], right=False, labels=BUCKETS
)

## Events

Job starts from memory (month precision; the 1st of the month is the cut-off).
*Confounded* = another life event (move-in / city move) within the same window —
shown separately, never pooled with the clean starters. `fluffy-beaver` (job
2020-09) is dropped: the export starts 2020-08-12, so there is no before-window.

The control for each starter = every author with **no event of their own** inside
that starter's window.

In [ ]:
JOB_STARTS = {
    "vibrant-barracuda": "2022-09",
    "striking-rail": "2023-01",
    "pliable-tiger": "2024-09",
    "animated-elk": "2022-09",
    "humorous-stingray": "2023-09",
    "effervescent-penguin": "2024-09",
    "rib-tickling-curlew": "2025-09",
}
CONFOUNDED = {"animated-elk", "humorous-stingray", "effervescent-penguin", "rib-tickling-curlew"}

# every other dated life event we know of (Analyses 9 and 10), used to keep controls clean
OTHER_EVENTS = {
    "striking-rail": ["2021-08", "2025-08"],
    "animated-elk": ["2022-09"],
    "effervescent-penguin": ["2023-07", "2024-07"],
    "humorous-stingray": ["2023-07"],
    "rib-tickling-curlew": ["2025-11"],
    "fluffy-beaver": ["2020-09", "2026-01"],
    "hypnotic-rabbit": ["2025-07"],
    "pliable-tiger": ["2026-08"],
}
EVENTS = {
    author: {pd.Timestamp(m + "-01") for m in OTHER_EVENTS.get(author, [])}
    | ({pd.Timestamp(JOB_STARTS[author] + "-01")} if author in JOB_STARTS else set())
    for author in weekdays["author"].unique()
}

In [ ]:
def bucket_profile(frame: pd.DataFrame) -> pd.Series:
    """Share of each active day's messages per bucket, averaged over days -- one day, one vote."""
    per_day = pd.crosstab(frame["date"], frame["bucket"], normalize="index")
    return per_day.reindex(columns=BUCKETS, fill_value=0).mean()


def controls_for(starter: str, start: pd.Timestamp, window: pd.DateOffset) -> list[str]:
    return [
        author for author, dates in EVENTS.items()
        if author != starter and not any(start - window <= d < start + window for d in dates)
    ]


def compare(starter: str, start: pd.Timestamp, months: int) -> dict:
    window = pd.DateOffset(months=months)
    groups = {
        "starter": weekdays[weekdays["author"] == starter],
        "control": weekdays[weekdays["author"].isin(controls_for(starter, start, window))],
    }
    result = {}
    for role, frame in groups.items():
        for period, (lo, hi) in {"before": (start - window, start), "after": (start, start + window)}.items():
            in_window = frame[(frame["timestamp"] >= lo) & (frame["timestamp"] < hi)]
            result[(role, period)] = bucket_profile(in_window)
            if role == "starter":
                result[("days", period)] = in_window["date"].nunique()
    return result


def work_difference(result: dict) -> float:
    starter_change = result[("starter", "after")][WORK] - result[("starter", "before")][WORK]
    control_change = result[("control", "after")][WORK] - result[("control", "before")][WORK]
    return starter_change - control_change

## Primary result (±6 months) and the ±12 sensitivity check

In [ ]:
results = {m: {w: compare(m, pd.Timestamp(d + "-01"), w) for w in (6, 12)} for m, d in JOB_STARTS.items()}

summary = pd.DataFrame([
    {
        "starter": m,
        "confounded": m in CONFOUNDED,
        "active_days_before": r[6][("days", "before")],
        "active_days_after": r[6][("days", "after")],
        "work_before": round(r[6][("starter", "before")][WORK], 3),
        "work_after": round(r[6][("starter", "after")][WORK], 3),
        "diff_6m": round(work_difference(r[6]), 3),
        "diff_12m": round(work_difference(r[12]), 3),
    }
    for m, r in results.items()
]).set_index("starter")
summary

## The chart

Each panel is one starter: the share of an active weekday's messages in each part of
the day, 6 months before (grey) vs. after (blue) the job start. Black ticks = the
control over the same months.

In [ ]:
GREY, BLUE, INK, MUTED = "#a3a29c", "#2a78d6", "#0b0b0b", "#52514e"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
})

def plot_starters(filename: str) -> None:
    x = np.arange(len(BUCKETS))
    fig, axes = plt.subplots(1, len(JOB_STARTS), figsize=(20, 3.8), sharey=True)
    for ax, starter in zip(axes, JOB_STARTS):
        r = results[starter][6]
        ax.bar(x - 0.21, r[("starter", "before")], 0.4, color=GREY, label="before job")
        ax.bar(x + 0.21, r[("starter", "after")], 0.4, color=BLUE, label="after job")
        ax.scatter(x - 0.21, r[("control", "before")], marker="_", s=260, color=INK, lw=2, zorder=3, label="control")
        ax.scatter(x + 0.21, r[("control", "after")], marker="_", s=260, color=INK, lw=2, zorder=3)
        ax.set_xticks(x, ["night", "morn.", "work", "eve."], fontsize=9)
        ax.set_title(starter + ("  (confounded)" if starter in CONFOUNDED else ""), fontsize=9)
    axes[0].set_ylabel("share of an active weekday's messages")
    axes[0].legend(fontsize=8, frameon=False, loc="upper left")
    fig.suptitle("Weekday messages by time of day, 6 months before vs. after starting a job", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

plot_starters("job-start-hour-of-day-v1.png")

## v2 — after critique round 1

- **Two rows:** clean starters on top, confounded below — they are different kinds of
  evidence, so they should not read as seven equal cases. No fading (it would look like
  hiding the confounded cases).
- **Colour only on the work-hours bucket** — the claim is about that bucket; the other
  three stay grey so the shape is still visible without competing for attention.
- **v2b** collapses to two buckets (work vs. not) for comparison — clearer, but the
  distribution becomes one number (non-work = 1 − work) and the shape is lost.

In [ ]:
CLEAN = [m for m in JOB_STARTS if m not in CONFOUNDED]
ROWS = {"clean": CLEAN, "confounded\n(job + move)": [m for m in JOB_STARTS if m in CONFOUNDED]}
LIGHT_GREY, LIGHT_BLUE = "#d6d5d0", "#a9c8ef"


def plot_v2(profiles_for, labels, highlight, filename: str, title: str) -> None:
    x = np.arange(len(labels))
    fig, axes = plt.subplots(2, 4, figsize=(15, 6.4), sharey=True)
    for row, (row_name, starters) in enumerate(ROWS.items()):
        axes[row, 0].set_ylabel(f"{row_name}\n\nshare of an active\nweekday's messages")
        for col in range(4):
            ax = axes[row, col]
            if col >= len(starters):
                ax.axis("off")
                continue
            before, after, c_before, c_after = profiles_for(starters[col])
            before_colours = [LIGHT_BLUE if b == highlight else LIGHT_GREY for b in labels]
            after_colours = [BLUE if b == highlight else GREY for b in labels]
            ax.bar(x - 0.21, before, 0.4, color=before_colours)
            ax.bar(x + 0.21, after, 0.4, color=after_colours)
            ax.scatter(x - 0.21, c_before, marker="_", s=240, color=INK, lw=2, zorder=3)
            ax.scatter(x + 0.21, c_after, marker="_", s=240, color=INK, lw=2, zorder=3)
            ax.set_xticks(x, labels, fontsize=8)
            ax.set_title(starters[col], fontsize=10, loc="left")
    handles = [
        plt.Rectangle((0, 0), 1, 1, color=LIGHT_BLUE), plt.Rectangle((0, 0), 1, 1, color=BLUE),
        plt.Line2D([], [], color=INK, marker="_", ls="", markersize=14, mew=2),
    ]
    axes[0, 3].legend(handles, ["6 months before job", "6 months after job", "control (others, same months)"],
                      frameon=False, loc="center left", fontsize=9)
    fig.suptitle(title, x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()


def four_buckets(m):
    r = results[m][6]
    return r[("starter", "before")], r[("starter", "after")], r[("control", "before")], r[("control", "after")]


def two_buckets(m):
    return tuple(pd.Series([p[WORK], 1 - p[WORK]]) for p in four_buckets(m))


TITLE_PLACEHOLDER = "Weekday messages by time of day, before vs. after starting a job  [title: student's claim, TBD]"
plot_v2(four_buckets, BUCKETS, WORK, "job-start-hour-of-day-v2.png", TITLE_PLACEHOLDER)
plot_v2(two_buckets, ["work hours\n09–17", "outside work hours"], "work hours\n09–17",
        "job-start-hour-of-day-v2b-two-buckets.png", TITLE_PLACEHOLDER)

## Verification — placebo job starts (Stage 6)

Agreed in Stage 1, before looking. For each starter, recompute the same statistic
at **fake job-start months**: every month where the ±6-month window fits inside the
export and contains **none of that person's real events**. That gives each person's
"normal" month-to-month wobble. Then ask whether the real start dates stand out.

- **Per starter:** where does the real difference sit among that person's placebos?
- **Combined:** the mean real difference over the starters vs. 2,000 draws of one random
  placebo per starter, averaged. Prediction was *negative*, so one-sided: how often is
  a placebo mean at least as negative as the real one? Done separately for the 3 clean
  starters (the primary claim) and for all 7.

In [ ]:
EXPORT_START, EXPORT_END = weekdays["timestamp"].min(), weekdays["timestamp"].max()
SIX = pd.DateOffset(months=6)


def placebo_months(starter: str) -> list[pd.Timestamp]:
    months = pd.date_range(EXPORT_START.normalize() + SIX, EXPORT_END - SIX, freq="MS")
    return [m for m in months if not any(m - SIX <= e < m + SIX for e in EVENTS[starter])]


placebos = {m: np.array([work_difference(compare(m, fake, 6)) for fake in placebo_months(m)]) for m in JOB_STARTS}
real = summary["diff_6m"]

per_starter = pd.DataFrame({
    "real_diff": real,
    "n_placebos": {m: len(v) for m, v in placebos.items()},
    "share_placebos_at_least_as_negative": {m: round((placebos[m] <= real[m]).mean(), 2) for m in JOB_STARTS},
})
per_starter

In [ ]:
rng = np.random.default_rng(42)


def combined_test(starters: list[str], draws: int = 2000) -> tuple[float, np.ndarray, float]:
    observed = real[starters].mean()
    null = np.array([np.mean([rng.choice(placebos[m]) for m in starters]) for _ in range(draws)])
    return observed, null, (null <= observed).mean()


tests = {"clean (3)": combined_test(CLEAN), "all (7)": combined_test(list(JOB_STARTS))}
for name, (observed, null, p_one_sided) in tests.items():
    print(f"{name:10s} observed mean diff = {observed:+.3f}   one-sided p = {p_one_sided:.2f}   "
          f"null 5-95%: [{np.quantile(null, 0.05):+.3f}, {np.quantile(null, 0.95):+.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, (name, (observed, null, p_one_sided)) in zip(axes, tests.items()):
    ax.hist(null, bins=40, color=LIGHT_GREY)
    ax.axvline(observed, color=BLUE, lw=2)
    ax.axvline(0, color=MUTED, lw=1, ls=":")
    ax.set_title(f"{name}: real = {observed:+.3f}, one-sided p = {p_one_sided:.2f}", loc="left", fontsize=10)
    ax.set_xlabel("mean change in work-hours share vs. control (placebo job starts)")
axes[0].set_ylabel("placebo draws")
fig.tight_layout()
fig.savefig("job-start-placebo-null.png", dpi=150, bbox_inches="tight")
plt.show()

## Final chart (v3)

Student's choice after critique: **v2b** (work hours vs. outside work hours) — the most
direct view of the one comparison. v2 is kept above as the version that still shows
*where* the non-work messages go. Title = the student's claim, set after the placebo test.

In [ ]:
plot_v2(two_buckets, ["work hours\n09–17", "outside work hours"], "work hours\n09–17",
        "job-start-hour-of-day-final.png",
        "People do not change the moment of texting after getting a job")

## Held-out slice — weekends (Stage 6)

A job only changes Mon–Fri. For the one starter who did shift (`vibrant-barracuda`),
the same statistic on **Sat–Sun** is a slice the analysis never used: if their 09–17
share drops at the weekend too, the weekday shift wasn't the job.

In [ ]:
weekends = own[own["timestamp"].dt.weekday >= 5].copy()
weekends["date"] = weekends["timestamp"].dt.normalize()
weekends["bucket"] = pd.cut(weekends["timestamp"].dt.hour, [0, 6, 9, 17, 24], right=False, labels=BUCKETS)

weekday_frame = weekdays
weekdays = weekends  # compare() reads the module-level frame
weekend_result = {m: compare(m, pd.Timestamp(d + "-01"), 6) for m, d in JOB_STARTS.items()}
weekdays = weekday_frame

pd.DataFrame({
    "weekday_diff": summary["diff_6m"],
    "weekend_diff": {m: round(work_difference(r), 3) for m, r in weekend_result.items()},
    "weekend_days_before": {m: r[("days", "before")] for m, r in weekend_result.items()},
    "weekend_days_after": {m: r[("days", "after")] for m, r in weekend_result.items()},
    "weekend_work_before": {m: round(r[("starter", "before")][WORK], 3) for m, r in weekend_result.items()},
    "weekend_work_after": {m: round(r[("starter", "after")][WORK], 3) for m, r in weekend_result.items()},
})